# Clase 8 · Práctica: Atención y Transformers

## Qué vas a practicar

En esta libreta aplicas lo que viste en la Clase 8 sobre atención y Transformers. Completas el código en dos ejercicios:

- **Ejercicio 1.** Implementas desde cero la atención escalada por producto punto (*scaled dot-product attention*), el bloque que compara una consulta con un conjunto de claves y combina los valores correspondientes según esa comparación.
- **Ejercicio 2.** Extiendes esa función a la atención multi-cabeza (*multi-head attention*): divides las consultas, claves y valores en varias cabezas, aplicas la atención del Ejercicio 1 a cada una por separado y concatenas los resultados.

Cada ejercicio incluye el andamiaje de código con partes marcadas como `# TODO`, que debes completar tú. Debajo de cada ejercicio encuentras una solución colapsada: intenta resolver el ejercicio por tu cuenta antes de abrirla.

## Requisito previo

Necesitas haber visto la Clase 8 ("Atención y Transformers"), en la que se explica el cuello de botella de las redes neuronales recurrentes (RNN) en secuencias largas, el mecanismo de atención con consulta, clave y valor (*query, key, value*), y la fórmula de la atención escalada por producto punto:

$$\text{Atención}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V$$

## Cómo ejecutar esta libreta

Ábrela en Google Colab y ejecuta las celdas en orden, de arriba a abajo. No instales nada adicional: `torch` ya viene preinstalado en Colab. No necesitas activar ningún acelerador: los tensores de este notebook son pequeños y se ejecutan rápido en el procesador estándar de Colab.

## Ejercicio 1 · Atención escalada por producto punto desde cero

### El caso

Formas parte del equipo técnico de un hub de innovación digital que prepara un taller interno sobre Transformers para las entidades socias. Antes de enseñar a nadie a usar `nn.MultiheadAttention`, quieres que el equipo entienda qué ocurre dentro de esa caja negra. Te encargas de programar, con las operaciones básicas de PyTorch, exactamente la fórmula que viste en la Clase 8: comparas una consulta con un conjunto de claves, conviertes esa comparación en pesos con softmax y usas esos pesos para combinar los valores.

### Lo que tienes que hacer

Completa el cuerpo de la función `atencion_escalada(Q, K, V)`. Recibe tres matrices —consultas, claves y valores— y devuelve la salida de la atención junto con los pesos de atención usados. El resto del andamiaje (los datos de ejemplo, la función `softmax_manual` y la celda que prueba la función) ya está escrito: no lo modifiques.

### Pistas

- La dimensión de las claves, $d_k$, es el tamaño de la última dimensión de `K`: obtenla con `K.size(-1)`.
- Multiplica `Q` por la traspuesta de `K` en sus dos últimas dimensiones con `K.transpose(-2, -1)`, no con `K.T` (así la función también funciona si más adelante añades una dimensión de lote).
- Escala el resultado dividiendo entre la raíz cuadrada de $d_k$, tal como viste en la teoría: `d_k ** 0.5`.
- Aplica `softmax_manual` sobre la última dimensión (`dim=-1`), para que cada fila de pesos sea una distribución de probabilidad que suma uno.
- La salida final es el producto de los pesos por `V`.

In [ ]:
import torch

torch.manual_seed(0)

# Frase de ejemplo, ya tokenizada en palabras
tokens = ["el", "hub", "conecta", "empresas", "con", "tecnologia", "avanzada"]
n = len(tokens)
d = 8    # dimensión del embedding, pequeña para que el ejemplo sea legible

# Embeddings de partida: en un modelo entrenado vendrían de una capa de
# embedding aprendida; aquí los generamos al azar para centrarnos en la
# mecánica de la atención
X = torch.randn(n, d)

# Proyecciones lineales aprendibles: transforman el mismo embedding en
# tres papeles distintos (consulta, clave, valor)
W_q = torch.randn(d, d) * 0.5
W_k = torch.randn(d, d) * 0.5
W_v = torch.randn(d, d) * 0.5

Q = X @ W_q
K = X @ W_k
V = X @ W_v


def softmax_manual(z, dim=-1):
    # Restamos el máximo antes de la exponencial por estabilidad numérica
    z = z - z.max(dim=dim, keepdim=True).values
    exp_z = torch.exp(z)
    return exp_z / exp_z.sum(dim=dim, keepdim=True)


def atencion_escalada(Q, K, V):
    """Calcula la atención escalada por producto punto.

    Devuelve una tupla (salida, pesos):
    - salida: combinación ponderada de V, misma forma que Q
    - pesos: pesos de atención, forma (n_consultas, n_claves)
    """
    d_k = K.size(-1)

    # TODO: calcula las puntuaciones de relevancia entre Q y K con producto
    # punto. Usa K.transpose(-2, -1) para trasponer las dos últimas
    # dimensiones de K antes de multiplicar
    puntuaciones = ...

    # TODO: escala las puntuaciones dividiendo entre la raíz cuadrada de d_k
    puntuaciones = ...

    # TODO: convierte las puntuaciones en pesos de atención con
    # softmax_manual, normalizando sobre la última dimensión
    pesos = ...

    # TODO: calcula la salida como combinación ponderada de V
    salida = ...

    return salida, pesos

In [ ]:
salida, pesos_atencion = atencion_escalada(Q, K, V)

print("Forma de la salida:", salida.shape)                     # (7, 8)
print("Cada fila de pesos suma:", pesos_atencion.sum(dim=-1))   # ~1.0 en cada posición

<details>
<summary>💡 Ver solución</summary>

```python
def atencion_escalada(Q, K, V):
    """Calcula la atención escalada por producto punto.

    Devuelve una tupla (salida, pesos):
    - salida: combinación ponderada de V, misma forma que Q
    - pesos: pesos de atención, forma (n_consultas, n_claves)
    """
    d_k = K.size(-1)

    # Producto punto entre cada consulta y cada clave
    puntuaciones = Q @ K.transpose(-2, -1)

    # Escalado por la raíz de d_k, para que el softmax no sature
    puntuaciones = puntuaciones / (d_k ** 0.5)

    # Cada fila se convierte en una distribución de probabilidad
    pesos = softmax_manual(puntuaciones, dim=-1)

    # Combinación ponderada de los valores
    salida = pesos @ V

    return salida, pesos
```
</details>

## Ejercicio 2 · Atención multi-cabeza

### El caso

La función que acabas de programar funciona, pero obliga al modelo a mezclar en un único patrón de pesos todo tipo de relaciones entre palabras: cercanía, concordancia gramatical, referencias... Para el mismo taller interno, el equipo te pide ahora el segundo bloque que viste en la Clase 8: la atención multi-cabeza. En lugar de una sola atención, ejecutas varias en paralelo —cada una sobre una porción distinta de la dimensión del embedding— y concatenas sus salidas.

### Lo que tienes que hacer

Completa la función `atencion_multi_cabeza(Q, K, V, n_cabezas)`. Reutiliza `atencion_escalada`, la función que implementaste en el Ejercicio 1. El andamiaje que divide `Q`, `K` y `V` en cabezas, y el que concatena las salidas al final, ya está escrito: tu tarea es aplicar `atencion_escalada` a cada cabeza dentro del bucle.

### Pistas

- `Q`, `K` y `V` llegan con forma `(n, d_modelo)`. Tras la división ya escrita, `Q_cabezas`, `K_cabezas` y `V_cabezas` tienen forma `(n_cabezas, n, d_cabeza)`, con `d_cabeza = d_modelo / n_cabezas`.
- Dentro del bucle, `Q_cabezas[i]`, `K_cabezas[i]` y `V_cabezas[i]` son las consultas, claves y valores de la cabeza i-ésima, con forma `(n, d_cabeza)`: exactamente lo que espera `atencion_escalada`.
- Llama a `atencion_escalada(Q_cabezas[i], K_cabezas[i], V_cabezas[i])` y guarda la salida en la lista `salidas_cabezas`, y los pesos en `pesos_cabezas`.
- No necesitas la matriz de proyección final $W^O$ de la teoría: para este ejercicio basta con concatenar las cabezas, tal como está escrito debajo del bucle.

In [ ]:
def atencion_multi_cabeza(Q, K, V, n_cabezas):
    """Aplica atencion_escalada en paralelo sobre varias cabezas y
    concatena los resultados.

    Q, K, V: tensores de forma (n, d_modelo)
    n_cabezas: número de cabezas en las que dividir d_modelo

    Devuelve una tupla (salida, pesos_cabezas):
    - salida: forma (n, d_modelo), igual que la entrada
    - pesos_cabezas: lista con los pesos de atención de cada cabeza
    """
    n, d_modelo = Q.shape
    assert d_modelo % n_cabezas == 0, "d_modelo debe ser divisible entre n_cabezas"
    d_cabeza = d_modelo // n_cabezas

    # Dividimos la última dimensión en n_cabezas trozos y movemos el eje
    # de cabeza al frente: (n, d_modelo) -> (n_cabezas, n, d_cabeza)
    Q_cabezas = Q.view(n, n_cabezas, d_cabeza).transpose(0, 1)
    K_cabezas = K.view(n, n_cabezas, d_cabeza).transpose(0, 1)
    V_cabezas = V.view(n, n_cabezas, d_cabeza).transpose(0, 1)

    salidas_cabezas = []
    pesos_cabezas = []
    for i in range(n_cabezas):
        # TODO: aplica atencion_escalada a la cabeza i-ésima
        # (Q_cabezas[i], K_cabezas[i], V_cabezas[i]) y añade la salida a
        # salidas_cabezas y los pesos a pesos_cabezas
        ...

    # Apilamos las cabezas y volvemos a la forma original (n, d_modelo)
    salida = torch.stack(salidas_cabezas, dim=0).transpose(0, 1).reshape(n, d_modelo)

    return salida, pesos_cabezas

In [ ]:
n_cabezas = 2
salida_mc, pesos_mc = atencion_multi_cabeza(Q, K, V, n_cabezas)

print("Forma de la salida:", salida_mc.shape)                   # (7, 8), igual que Q
print("Número de cabezas devueltas:", len(pesos_mc))             # 2
print("Forma de los pesos de una cabeza:", pesos_mc[0].shape)    # (7, 4)

<details>
<summary>💡 Ver solución</summary>

```python
def atencion_multi_cabeza(Q, K, V, n_cabezas):
    """Aplica atencion_escalada en paralelo sobre varias cabezas y
    concatena los resultados."""
    n, d_modelo = Q.shape
    assert d_modelo % n_cabezas == 0, "d_modelo debe ser divisible entre n_cabezas"
    d_cabeza = d_modelo // n_cabezas

    Q_cabezas = Q.view(n, n_cabezas, d_cabeza).transpose(0, 1)
    K_cabezas = K.view(n, n_cabezas, d_cabeza).transpose(0, 1)
    V_cabezas = V.view(n, n_cabezas, d_cabeza).transpose(0, 1)

    salidas_cabezas = []
    pesos_cabezas = []
    for i in range(n_cabezas):
        salida_i, pesos_i = atencion_escalada(
            Q_cabezas[i], K_cabezas[i], V_cabezas[i]
        )
        salidas_cabezas.append(salida_i)
        pesos_cabezas.append(pesos_i)

    salida = torch.stack(salidas_cabezas, dim=0).transpose(0, 1).reshape(n, d_modelo)

    return salida, pesos_cabezas
```
</details>

## Para terminar

Acabas de construir, con operaciones básicas de tensores, las dos piezas que sostienen cualquier Transformer moderno: la atención escalada por producto punto y su versión multi-cabeza. Todo lo demás —la normalización, la red hacia adelante, el apilamiento de bloques— se apoya directamente sobre lo que acabas de programar aquí.

Antes de la próxima clase en vivo, prueba a cambiar `n_cabezas` a cuatro (usa entonces un embedding de dimensión mayor, por ejemplo 16, para que la división siga siendo exacta) y observa cómo cambia la forma de los pesos de cada cabeza.

**¡Comparte tu solución y tus dudas en el canal del curso antes de la próxima clase en vivo!**